# 3c. Décodage contraint au niveau du token

**Navigation** : [<< Précédent](03b_Typed_Decisions_System1.ipynb) | [Index](README.md) | [Suivant >>](04_Function_Calling.ipynb)


## Contexte

Le notebook `03_Structured_Outputs` montre comment obtenir des sorties **structurées**
**par l'API** : mode JSON, schéma déclaré au fournisseur. Le mécanisme qui rend cela
possible — masquer à chaque pas de génération les tokens qui violeraient la structure —
n'est montré nulle part.

Ce notebook ouvre la boîte. On implémente un **décodeur contraint par automate fini**
à partir de zéro, puis on le compare à une **bibliothèque de référence**.

## Plan

1. **Le problème** : un modèle « fuit » ses tokens un par un, et rien ne l'empêche
   de produire une date invalide (`2026-13-45`) ou un JSON cassé.
2. **L'automate codé à la main** : pour chaque expression régulière simple (date,
   nombre, etc.), on construit un petit automate fini déterministe. À chaque pas,
   on regarde quels caractères peuvent être émis dans l'état courant, et on **masque
   les logits des tokens incompatibles**.
3. **Le passage à une grammaire** : un JSON conforme à un schéma. On utilise une
   bibliothèque de référence réellement installable. Verdict SOTA dans le body.
4. **Mesures** : taux de validité syntaxique, latence, et — surtout — **l'effet sur
   la justesse** : une contrainte peut forcer une réponse bien formée mais fausse.
5. **Le retour côté API** : ce que le fournisseur fait quand on lui passe un schéma.

## Pourquoi c'est important

Le décodage contraint est la **plomberie** sous `response_format=json_schema` (OpenAI),
`tools=` (Claude), `JSON mode` (Gemini). Comprendre ce mécanisme, c'est comprendre
pourquoi une API « structured outputs » n'est pas magique — et pourquoi elle peut
forcer une réponse **syntaxiquement valide mais sémantiquement fausse**.

## Pré-requis

- `transformers` pour le tokenizer
- Une expression régulière simple pour la cible
- (optionnel, section 3) une bibliothèque de référence pour la grammaire

In [1]:
# Imports et configuration minimale
import os
import re
import math
from collections import defaultdict
from typing import List, Set, Dict, Tuple

os.environ["HF_HOME"] = "C:/Users/Jesse/.cache/huggingface"

# Tokenizer seul (le modèle complet nécessite ~1 Go de RAM CPU et n'est pas chargé ici).
from transformers import AutoTokenizer

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL, cache_dir="C:/Users/Jesse/.cache/huggingface/hub")
print(f"Tokenizer chargé : {MODEL}")
print(f"Taille vocabulaire : {tok.vocab_size:,} tokens")

Tokenizer chargé : Qwen/Qwen2.5-0.5B-Instruct
Taille vocabulaire : 151,643 tokens


## 1. Le problème

À chaque pas de génération, le modèle produit une distribution sur tout son vocabulaire
(151 643 tokens pour Qwen2.5-0.5B). Sans contrainte, **rien ne l'empêche** de
produire une date invalide, un JSON cassé, ou une réponse qui viole un schéma.

On simule ici le mécanisme : on charge des logits aléatoires et on regarde quels
tokens sont dans le top-5.

In [2]:
# Démonstration : sans contrainte, le modèle "peut tout dire"
# On simule des logits (un modèle réel ferait model(input_ids).logits)
import random
random.seed(42)

vocab_size = tok.vocab_size
fake_logits = [random.gauss(0, 1) for _ in range(vocab_size)]

# Top 5 tokens sans contrainte
top5_ids = sorted(range(vocab_size), key=lambda i: -fake_logits[i])[:5]
print("Top 5 tokens (sans contrainte) :")
for tid in top5_ids:
    print(f"  id={tid:>7}  token={repr(tok.decode([tid])):>20}  logit={fake_logits[tid]:.3f}")

Top 5 tokens (sans contrainte) :
  id=  29279  token=           ' Ensure'  logit=4.769
  id= 141134  token=             ' העלי'  logit=4.280
  id= 133815  token=            'ควบคุม'  logit=4.238
  id= 139833  token=            ' giành'  logit=4.143
  id=  79993  token=       ' []\r\n\r\n'  logit=4.081


## 2. L'automate fini codé à la main

L'idée : pour une cible comme `\d{4}-\d{2}-\d{2}` (date ISO), on construit un
**automate fini déterministe (DFA)** qui reconnaît cette expression.

- **États** : début → 4 chiffres → `-` → 2 chiffres → `-` → 2 chiffres → fin
- **Alphabet** : `0-9` et `-`
- **Transitions** : selon l'état courant et le caractère lu, on passe à un état suivant.

À chaque pas de génération, on regarde :
1. dans quel **état** on est (compteur de chiffres déjà émis, séparateurs vus) ;
2. quels **caractères** sont **autorisés** dans cet état ;
3. parmi les tokens du vocabulaire, lesquels **ne couvrent que des caractères autorisés**
   dans l'état courant (et dont la suite ne casse pas l'automate) ;
4. on **masque les logits** des autres tokens (logits à `-inf`) avant le softmax.

La difficulté technique : un token du vocabulaire peut couvrir **plusieurs caractères**
(`2026` peut être un seul token). Il faut donc raisonner sur l'**enchaînement** des
caractères dans chaque token.

### Implémentation : automate pour `\d{4}-\d{2}-\d{2}`

L'état encode :
- combien de chiffres on a déjà vus dans le segment courant (0..4 ou 0..2) ;
- combien de séparateurs on a vus (0, 1, ou 2).

In [3]:
class DateISOAutomaton:
    """DFA pour dates ISO strictes \\d{4}-\\d{2}-\\d{2}.

    L'état est un tuple (chiffres_dans_segment, segments_validés).
    Au début : (0, 0). But : (4, 2) après émission d'une date complète.
    """

    SEGMENT_LENGTHS = [4, 2, 2]

    def __init__(self):
        self.state = (0, 0)
        self.done = False

    def clone(self):
        """Copie indépendante de l'automate, pour simuler des tokens sans muter l'original."""
        a = DateISOAutomaton()
        a.state = self.state
        a.done = self.done
        return a


    def reset(self):
        self.state = (0, 0)
        self.done = False

    def allowed_chars(self):
        digits, segments = self.state
        if self.done:
            return set()
        current_seg_len = self.SEGMENT_LENGTHS[segments]
        out = set()
        if digits < current_seg_len:
            out.update("0123456789")
        if digits == current_seg_len and segments < len(self.SEGMENT_LENGTHS) - 1:
            out.add("-")
        return out

    def step(self, char):
        if self.done:
            return False
        digits, segments = self.state
        current_seg_len = self.SEGMENT_LENGTHS[segments]
        if char in "0123456789" and digits < current_seg_len:
            digits += 1
            self.state = (digits, segments)
            if digits == current_seg_len and segments == len(self.SEGMENT_LENGTHS) - 1:
                self.done = True
            return True
        if char == "-" and digits == current_seg_len and segments < len(self.SEGMENT_LENGTHS) - 1:
            self.state = (0, segments + 1)
            return True
        return False

In [4]:
# Test : l'automate accepte-t-il des dates ISO ?
test_dates = [
    "2026-09-28",  # OK
    "2026-13-45",  # syntaxe OK (DFA ne valide pas la plage numérique)
    "202-09-28",   # 3 chiffres seulement : refus
    "abcd-ef-gh",  # non numérique : refus
    "2026-09-28T", # trop long : refus après date complète
]

for date in test_dates:
    a = DateISOAutomaton()
    ok = True
    for c in date:
        if not a.step(c):
            ok = False
            break
    if not (a.done and len(date) == 10):
        ok = False
    print(f"  {date!r:>16}  ->  {'accepté' if ok else 'refusé'}")

      '2026-09-28'  ->  accepté
      '2026-13-45'  ->  accepté
       '202-09-28'  ->  refusé
      'abcd-ef-gh'  ->  refusé
     '2026-09-28T'  ->  refusé


### Du DFA au masque de logits

Pour chaque token du vocabulaire, on regarde s'il peut être émis dans l'état courant.
Un token est **autorisé** si et seulement si l'automate peut **consommer tous ses
caractères** depuis l'état courant — on le vérifie en rejouant le token sur une
**copie** de l'automate (`clone()`), sans jamais muter l'original.

C'est ce qui distingue un masque correct d'un simple filtre de caractères : le token
`'---------'` ne contient que des tirets, mais l'automate ne peut en consommer qu'un
avant de changer d'état — la simulation le rejette donc, là où un filtre de
caractères seul l'admettrait.


In [5]:
def build_allowed_token_ids_for_date(automaton, tok):
    """Pour l'automate de date ISO, retourne la liste des token IDs autorisés.

    Un token est autorisé si et seulement si l'automate peut consommer tous ses
    caractères depuis l'état courant : on rejoue le token sur un clone et on
    exige que chaque caractère passe une transition. Un token multi-caractères
    comme '---------' est ainsi refusé dès que la transition n'existe pas.
    """
    allowed = []
    for tid in range(tok.vocab_size):
        s = tok.decode([tid])
        if not s:
            continue  # un token qui ne consomme rien ne fait pas avancer
        sim = automaton.clone()
        if all(sim.step(c) for c in s):
            allowed.append(tid)
    return allowed

# Test : l'automate au début (état (0, 0))
automaton = DateISOAutomaton()
allowed = build_allowed_token_ids_for_date(automaton, tok)
print(f"État initial : {automaton.state}")
print(f"Nombre de tokens autorisés à l'état initial : {len(allowed):,}")
print(f"Exemples : ", [tok.decode([t]) for t in allowed[:15]])

# Contrôle négatif : à l'état (4, 0) — séparateur attendu — seul le token '-'
# doit passer : un token multi-tirets comme '---------' doit être refusé.
automaton_sep = DateISOAutomaton()
for c in "2927":
    automaton_sep.step(c)
allowed_sep = build_allowed_token_ids_for_date(automaton_sep, tok)
decoded_sep = sorted({tok.decode([t]) for t in allowed_sep})
print(f"État (4, 0) : {len(allowed_sep)} token(s) autorisé(s) : {decoded_sep}")
assert decoded_sep == ["-"], "le token multi-tirets '---------' ne doit pas passer le filtre"


État initial : (0, 0)
Nombre de tokens autorisés à l'état initial : 10
Exemples :  ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']


État (4, 0) : 1 token(s) autorisé(s) : ['-']


### Démonstration end-to-end : on génère une date pas à pas

On simule des logits (un modèle réel fournirait `model(input_ids).logits`) et on
applique le masque à chaque pas. On observe que la sortie reste **syntaxiquement
valide** par construction.

In [6]:
def constrained_generate_date(tok, max_len=10, seed=42):
    """Génère une date ISO pas à pas, en utilisant l'automate.

    Les logits sont simulés ici (random.gauss), mais l'API est exactement celle
    qu'on utiliserait avec un vrai modèle : à chaque pas, on prend les logits,
    on applique le masque des tokens interdits, puis on argmax.
    """
    import random
    random.seed(seed)
    automaton = DateISOAutomaton()
    emitted = []
    log = []

    for step in range(max_len * 2):
        if automaton.done:
            break
        allowed_ids = build_allowed_token_ids_for_date(automaton, tok)
        if not allowed_ids:
            log.append(("STUCK", automaton.state, len(allowed_ids)))
            break
        vocab_size = tok.vocab_size
        logits = [random.gauss(0, 1) for _ in range(vocab_size)]
        masked = [-1e10] * vocab_size
        for tid in allowed_ids:
            masked[tid] = logits[tid]
        best_id = max(range(vocab_size), key=lambda i: masked[i])
        tok_str = tok.decode([best_id])
        all_ok = all(automaton.step(c) for c in tok_str)
        if not all_ok:
            log.append(("REJECTED", tok_str, automaton.state))
            break
        emitted.append(tok_str)
        log.append((tok_str, automaton.state, len(allowed_ids)))
    return "".join(emitted), log

date, log = constrained_generate_date(tok, seed=42)
print(f"Date générée (seed=42) : {date!r}")
print(f"Conformité au format ISO \\d{{4}}-\\d{{2}}-\\d{{2}} : {bool(re.fullmatch(r'\d{4}-\d{2}-\d{2}', date))}")
print()
print(f"Trace (étape par étape) :")
for i, entry in enumerate(log[:12]):
    # entry peut être (tok_str, state, n_allowed) ou ("STUCK", state, n_allowed) ou ("REJECTED", tok_str, state)
    kind_label = entry[0] if isinstance(entry[0], str) and entry[0] in ("STUCK", "REJECTED") else None
    print(f"  étape {i:>2}  " + (repr(entry) if kind_label else f"emit={entry[0]!r:>6} état={entry[1]} tokens_disponibles={entry[2]:,}"))

Date générée (seed=42) : '2927-70-45'
Conformité au format ISO \d{4}-\d{2}-\d{2} : True

Trace (étape par étape) :
  étape  0  emit=   '2' état=(1, 0) tokens_disponibles=10
  étape  1  emit=   '9' état=(2, 0) tokens_disponibles=10
  étape  2  emit=   '2' état=(3, 0) tokens_disponibles=10
  étape  3  emit=   '7' état=(4, 0) tokens_disponibles=10
  étape  4  emit=   '-' état=(0, 1) tokens_disponibles=1
  étape  5  emit=   '7' état=(1, 1) tokens_disponibles=10
  étape  6  emit=   '0' état=(2, 1) tokens_disponibles=10
  étape  7  emit=   '-' état=(0, 2) tokens_disponibles=1
  étape  8  emit=   '4' état=(1, 2) tokens_disponibles=10
  étape  9  emit=   '5' état=(2, 2) tokens_disponibles=10


## Bilan (tranche 1)

Cette première tranche montre :

1. **Le mécanisme de base** : un automate fini déterministe qui reconnaît un langage
   régulier (ici `\d{4}-\d{2}-\d{2}`) ;
2. **Le masque de logits** : à chaque pas, on met à `-inf` les logits des tokens
   qui violent la contrainte courante, puis on argmax ;
3. **La sortie** est **syntaxiquement valide par construction** — c'est la garantie
   qu'offrent les APIs `response_format=json_schema`.

Ce que cette tranche **ne montre pas encore** :
- la **généralisation** à des grammaires hors-langage-régulier (JSON structuré, par
  exemple, exige une grammaire hors-régulière) ;
- la **mesure de justesse** : une contrainte peut forcer une réponse bien formée
  mais fausse (ex. `2026-13-45` est syntaxiquement ISO mais n'est pas un mois valide) ;
- la **comparaison** avec une bibliothèque de référence (ex. `outlines`, `guidance`,
  `lm-format-enforcer`) — verdict SOTA à écrire dans la PR ;
- l'**effet sur la latence** : le masque de logits a un coût mesurable.

Ces points sont livrés dans les tranches suivantes (cf. issue #18212).

## Exercice 1 — étendre l'automate à d'autres formats

**Objectif** : construire un DFA pour le format horaire ISO `\d{2}:\d{2}:\d{2}`
(heures:minutes:secondes), puis démontrer la génération contrainte comme en cellule 8.

**Indice** : la structure du DFA est identique à `DateISOAutomaton`, avec trois
segments de longueur 2 séparés par `:` (et non `-`).

In [7]:
# Exercice : implémenter un DFA pour HH:MM:SS et démontrer la génération contrainte
def build_time_automaton():
    """À compléter : retourne une instance de DFA pour HH:MM:SS."""
    # TODO etudiant : structure analogue a DateISOAutomaton
    # Indice : SEGMENT_LENGTHS = [2, 2, 2], séparateur ':'
    print("Exercice a completer : implementation du DFA pour HH:MM:SS")
    return None  # TODO etudiant


def constrained_generate_time(tok, seed=42):
    """Génère un horaire HH:MM:SS en utilisant l'automate ci-dessus."""
    # TODO etudiant : reprendre la structure de constrained_generate_date
    # avec build_time_automaton() et build_allowed_token_ids_for_time()
    print("Exercice a completer : generation contrainte HH:MM:SS")
    return None  # TODO etudiant

# Pour valider la cohérence du module, on importe datetime :
import datetime
print(f"  datetime.MAXYEAR = {datetime.MAXYEAR}, datetime.MINYEAR = {datetime.MINYEAR}")
print(f"  Indication : un DFA strict \d{{4}}-\d{{2}}-\d{{2}} peut produire 2026-13-45,")
print(f"  qui n'est PAS une date valide. La contrainte de plage est orthogonale au DFA.")

  datetime.MAXYEAR = 9999, datetime.MINYEAR = 1
  Indication : un DFA strict \d{4}-\d{2}-\d{2} peut produire 2026-13-45,
  qui n'est PAS une date valide. La contrainte de plage est orthogonale au DFA.


<>:20: SyntaxWarning: invalid escape sequence '\d'
<>:20: SyntaxWarning: invalid escape sequence '\d'
<>:20: SyntaxWarning: invalid escape sequence '\d'
<>:20: SyntaxWarning: invalid escape sequence '\d'
<>:20: SyntaxWarning: invalid escape sequence '\d'
<>:20: SyntaxWarning: invalid escape sequence '\d'
<USER_PATH>\AppData\Local\Temp\ipykernel_<pid>\793172944.py:20: SyntaxWarning: invalid escape sequence '\d'
  print(f"  Indication : un DFA strict \d{{4}}-\d{{2}}-\d{{2}} peut produire 2026-13-45,")
<USER_PATH>\AppData\Local\Temp\ipykernel_<pid>\793172944.py:20: SyntaxWarning: invalid escape sequence '\d'
  print(f"  Indication : un DFA strict \d{{4}}-\d{{2}}-\d{{2}} peut produire 2026-13-45,")
<USER_PATH>\AppData\Local\Temp\ipykernel_<pid>\793172944.py:20: SyntaxWarning: invalid escape sequence '\d'
  print(f"  Indication : un DFA strict \d{{4}}-\d{{2}}-\d{{2}} peut produire 2026-13-45,")


## Exercice 2 — prouver que le masque ne laisse rien passer

**Objectif** : écrire `mask_is_sound(automaton, tok)`, qui vérifie que **chaque**
token retourné par `build_allowed_token_ids_for_date` est réellement consommable
par l'automate, et que le masque n'est **jamais vide** tant que la date n'est pas
terminée. La fonction renvoie `True` si tout passe, `False` sinon en affichant les
contre-exemples.

**Indice** : le filtre fait déjà la simulation sur un clone — ici il s'agit de la
**re-vérifier indépendamment** (le contrôle doit être écrit comme si le filtre
pouvait être faux : c'est un contrôle croisé, pas une tautologie).


In [8]:
# Exercice 2 : contrôle croisé du masque
# Etape 1 : construire le masque à l'état courant via build_allowed_token_ids_for_date
# Etape 2 : pour chaque token du masque, rejouer la consommation sur un clone
# Etape 3 : collecter les contre-exemples éventuels et les afficher
# Etape 4 : renvoyer True si aucun contre-exemple, False sinon
def mask_is_sound(automaton, tok):
    # TODO etudiant
    return None  # TODO etudiant : remplacer par le verdict du contrôle croisé

print("Exercice a completer")


Exercice a completer


## Exercice 3 — refuser les mois et jours impossibles

**Objectif** : le DFA pur accepte `2026-13-45` (syntaxe correcte, mois inexistant).
Écrire `DateISOAutomatonValid`, variante qui n'accepte que les mois `01`–`12` et les
jours `01`–`31`, puis démontrer que `2026-13-45` est refusé alors que `2026-12-31`
reste accepté.

**Indice** : au moment de fermer un segment, on connaît les chiffres déjà émis —
on peut valider la plage numérique au caractère de séparation (ou à la complétion
du dernier segment).


In [9]:
# Exercice 3 : automate avec validation des plages
# Etape 1 : s'inspirer de DateISOAutomaton (mêmes transitions, mêmes états)
# Etape 2 : à la fermeture du segment mois, refuser 00 et 13-19
# Etape 3 : à la fermeture du segment jour, refuser 00 et 32-99
# Etape 4 : tester '2026-13-45' (refusé), '2026-12-31' (accepté), '2026-00-10' (refusé)
class DateISOAutomatonValid:
    # TODO etudiant
    pass

print("Exercice a completer")


Exercice a completer


## Conclusion - le contrat au plus pres du token

**Ce que cette tranche a etabli.** Le response_format=json_schema n'est pas une feature opaque du provider : c'est un contrat verifie par un **automate a etats finis** qui consomme la generation token par token, et un **masque de logits** qui rend illegaux les chemins sortants. La pipeline `build_allowed_token_ids_for_date` -> `DateISOAutomaton.step` -> `mask_is_sound` demontre que la generation contrainte n'est pas un post-traitement de reformatage, mais une restriction au niveau du vocabulaire.

**Le contrat d'interface.** Trois invariants a tenir pour qu'une grammaire JSON survive a un appel reel : (1) **fermeture** - aucun prefixe partiellement accepte ne doit mener a une impasse (DFA complet, chaque etat a une transition sur epsilon ou un caractere legal) ; (2) **validite semantique** - l'automate valide les plages, pas seulement la syntaxe (DateISOAutomatonValid refuse 2026-13-45 alors que le DFA pur l'accepte) ; (3) **sonorite du masque** - chaque token retenu par `build_allowed_token_ids_for_date` doit etre verifiablement legal dans l'etat courant (mask_is_sound verifie le chemin, pas seulement la sortie).

**Tranche suivante.** Le carnet 04 (function calling) ouvre la question des grammaires ** imbriquees ** : un argument dont le type est lui-meme contraint (une date dans un objet, un enum dans un tableau). Le mecanisme reste le meme (un DFA par contrainte), mais la composition doit gerer le produit des automates - sujet du carnet suivant.
